In [2]:
import os
import json
import shutil

import pandas as pd
from glob import glob
from Bio import SeqIO

In [3]:
%cd ../analysis/2.Structure_prediction/

[Errno 2] No such file or directory: '../analysis/2.Structure_prediction/'
/var2/Works/junhyeong/RXLR_landscape/analysis/2.Structure_prediction


/home/junhyeong/miniconda3/envs/biopython-env/lib/python3.13/site-packages/IPython/core/magics/osm.py:393: UserWarning: This is now an optional IPython functionality, using bookmarks requires you to install the `pickleshare` library.
  bkms = self.shell.db.get('bookmarks', {})


In [4]:
entries = []
seqs = []
plddt = []
ptm = []

In [5]:
for record in SeqIO.parse("RXLR_fasta_v3.fasta", "fasta"):
    entry = record.id
    seq = str(record.seq)

    assert os.path.exists(f"pdb/{entry}.pdb")
    with open(f"json/{entry}.json") as f:
        data = json.load(f)

    plddt.append(sum(data["plddt"]) / len(data["plddt"]))
    ptm.append(data["ptm"])

    entries.append(entry)
    seqs.append(seq)

In [6]:
df = pd.DataFrame()

In [7]:
df["entry"] = entries
df["seq"] = seqs
df["plddt"] = plddt
df["ptm"] = ptm

In [8]:
df

,entry,seq,plddt,ptm
0,TDH64719.1,VAMTTDVVMHGGIVGKLKELLPNNDISNPFSKTRPSKTVTNSYGFN...,33.366736,0.24
1,TDH64798.1,ELFLKRSNVPFSIREAYNRNLQEHMTYLEDVNMYARQVRHEFSYDM...,88.604146,0.82
2,TDH65555.1,YGWRYPLRYWNRFGRPLFGSNCRFGRPWVGQLDVARCDDEKKEAEK...,67.174348,0.35
3,TDH65737.1,YGWRYPLRYWNRKKERGEKEAFDVMTSSFMHIE,80.266667,0.37
4,TDH65748.1,NKGFIYTLNKLHVWNMLEYERVVYLDADNILIRNSDELFLCGEFCA...,76.312921,0.69
...,...,...,...,...
4133,PYU1_T007635,KDGTVDLVIELAQTTESTLESIQETAFSSRTAKVEAIKSKLQAQSK...,95.434218,0.95
4134,PYU1_T002439,SDSGSGSVHEDENKTATHIFKSASDVKVGPGIVAAGAIVVGLVVCL...,76.487947,0.66
4135,PYU1_T003553,QQGFVNLIVILADTTESTLESIQESEYATRTDRINALKSKLEAGNK...,96.082014,0.95
4136,PYU1_T003569,VQGNYGGGGGATQSTGGASMSAGASGGAYDSSSGKTSSSGVKSSSS...,41.648927,0.12


In [9]:
df.to_csv("RXLR_fasta_v3.tsv", sep = "\t", index = False)

In [10]:
df[df["plddt"] >= 50].to_csv("RXLR_fasta_v3.qc.tsv", sep = "\t", index = False)

In [11]:
with open("RXLR_fasta_v3_qc.fasta", "w") as f:
    for idx in df.index:
        entry = df.loc[idx, "entry"]
        seq = df.loc[idx, "seq"]
        plddt = df.loc[idx, "plddt"]
    
        if plddt >= 50:
            f.write(f">{entry}\n{seq}\n")

            shutil.copyfile(f"pdb/{entry}.pdb", f"pdb_qc/{entry}.pdb")
            shutil.copyfile(f"json/{entry}.json", f"json_qc/{entry}.json")